## Convert NEUROVASC into MEDS

In [ ]:
import os
import joblib

ETL_OUTPUT = "MEDS_cohort"
ETL_INTERMEDIATE = "pre_MEDS"
ETL_INPUT = "raw_input"
EXPORT_DIR = "exports"
ETL_LABELS = f"{EXPORT_DIR}/labels"

os.makedirs(ETL_INPUT, exist_ok=True)
os.makedirs(ETL_INTERMEDIATE, exist_ok=True)
os.makedirs(ETL_OUTPUT, exist_ok=True)
os.makedirs(ETL_LABELS, exist_ok=True)

NUM_PATIENTS = 10000
TIME_OPT = "TS"
SYN_NEUROVASC_DATA = "https://raw.githubusercontent.com/TeamHeKA/neurovasc/refs/heads/main/exp/data/syn_data_10000.csv"

### a) Load/Generate source dataset

In [ ]:
import pandas as pd

# from NEUROVASC.utils.synthetic_generator import generate_synthetic_dataset
# df_input = generate_synthetic_dataset(NUM_PATIENTS, output_csv=f"{ETL_INPUT}/syn_data.csv")
df_input = pd.read_csv(SYN_NEUROVASC_DATA, index_col=0)
df_input = df_input.sample(frac=1, random_state=42).reset_index(drop=True)
df_input = df_input.rename(columns={"output": "outcome"})

### b) Preprocess the source dataset

In [ ]:
from utils.pre_MEDS import generate_meds_preprocessed

joblib.dump(
    df_input["outcome"].astype(int).to_list(),
    f"{ETL_LABELS}/outcomes_meds_{TIME_OPT}_{NUM_PATIENTS}.joblib",
)

df_input_outcome = df_input.iloc[0:NUM_PATIENTS]
df_input_no_outcome = df_input_outcome.drop(columns=["outcome"])
generate_meds_preprocessed(df_input_no_outcome, output_path=ETL_INTERMEDIATE)

print("Neurovasc data ready for MEDS-Extract")

### c) Run MEDS_Extract ETL to convert source into MEDS

In [ ]:
from MEDS_transforms.runner import main
import shutil

shutil.rmtree(ETL_OUTPUT)

main(
    [
        "pkg://MEDS_extract.configs._extract.yaml",
        "--overrides",
        f"input_dir={ETL_INTERMEDIATE}",
        f"output_dir={ETL_OUTPUT}",
        "event_conversion_config_fp=MESSY.yaml",
        "dataset.name=Neurovasc",
        "dataset.version=1.0",
    ]
)

### d) Convert NEUROVASC_MEDS into KG through MEDS2RDF

In [ ]:
from pathlib import Path

from meds2rdf import MedsRDFConverter
from meds2rdf.sinks import NTriplesSink
from meds2rdf.config import Config, MEDSSchema


engine = MedsRDFConverter(ETL_OUTPUT)
export_dir = Path(EXPORT_DIR) / f"meds_{NUM_PATIENTS}"

engine.convert(
    sink=NTriplesSink(export_dir, gzip_mode=False),
    cfg=Config(schemas={MEDSSchema.CODES}),
)

# sink = GraphSink(graph=Graph())
# sink.graph.serialize(f"{ETL_GRAPH}/meds_{TIME_OPT}_{NUM_PATIENTS}.nt", format="nt")
# shacl_graph = "https://raw.githubusercontent.com/TeamHeKA/meds-ontology/refs/heads/main/shacl/meds-shapes.ttl"

    ### e) Predict patient outcomes with tabular-based models

In [ ]:
from utils.tabular import run_tabulars_models

X = run_tabulars_models(
    meds_root=ETL_OUTPUT,
    classes=["BackHome", "Rehab", "Death"],
    outcomes_path=f"{ETL_LABELS}/outcomes_meds_{TIME_OPT}_{NUM_PATIENTS}.joblib",
    result_dir=f"{EXPORT_DIR}/metrics_{NUM_PATIENTS}",
)

### f) Compute metrics [OPTIONAL]

In [ ]:
from NEUROVASC.utils.metrics import (
    compute_MEDS_graph_metrics_for_neurovasc,
    save_stats_json,
)
from pathlib import Path
import glob
import gzip
from rdflib import Graph

g = Graph()

for path in glob.glob(f"{EXPORT_DIR}/*.nt.gz"):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        g.parse(f, format="nt")

stats = compute_MEDS_graph_metrics_for_neurovasc(
    MEDS_ETL_output_path=ETL_OUTPUT,
    graph=g,
    tabular_data=df_input_no_outcome,
    MEDS_intermediate=Path(ETL_INTERMEDIATE),
)

save_stats_json(stats, f"{ETL_OUTPUT}/metrics/all_metrics_2.json")

### g) Compute consistency checks [OPTIONAL]

In [ ]:
import polars as pl
from NEUROVASC.utils.neurovasc_meta import EVENTS_COLUMNS
from NEUROVASC.utils.transformers import (
    build_neurovasc_meds_dt,
    build_neurovasc_medskg_dt,
    check_dts_consistency,
)

synt_df = df_input.copy()
synt_df[EVENTS_COLUMNS] = (synt_df[EVENTS_COLUMNS] > -1).astype(int)

mimic_outcomes = joblib.load(
    f"{ETL_LABELS}/outcomes_meds_{TIME_OPT}_{NUM_PATIENTS}.joblib"
)

meds_data = pl.read_parquet(str(f"{ETL_OUTPUT}/data/**/*.parquet")).to_dicts()
meds_df = build_neurovasc_meds_dt(meds_data)
meds_df = meds_df.astype(synt_df.dtypes.to_dict())
meds_df["outcome"] = mimic_outcomes

graph_df = build_neurovasc_medskg_dt(g)
graph_df = graph_df.astype(synt_df.dtypes.to_dict())
graph_df["outcome"] = mimic_outcomes


def remove_digits(_df: pd.DataFrame):
    _df["hospital_stay_length"] = _df["hospital_stay_length"].round()
    _df["nb_acte"] = _df["nb_acte"].round()
    _df["age"] = _df["age"].round()
    _df["gcs"] = _df["gcs"].round(2)


for _df in [synt_df, meds_df, graph_df]:
    remove_digits(_df)

check_dts_consistency(meds_df, synt_df)
check_dts_consistency(meds_df, graph_df)
check_dts_consistency(graph_df, synt_df)
